# Run Large Language Models (LLMs)

This demo runs a privately hosted LLM: Qwen3-14B.

In [ ]:
import json
import requests

service_name = "huggingface-qwen3-14b"
namespace = "kserve-test"
ingress_domain = "svc.cluster.local"
gateway_host = "knative-local-gateway.istio-system.svc.cluster.local"
gateway_port = 80
model_name = "qwen3-14B"

host_header = f"{service_name}.{namespace}.{ingress_domain}"
headers = {
    "Host": host_header,
    "Content-Type": "application/json",
}

url = f"http://{gateway_host}:{gateway_port}/openai/v1/chat/completions"

In [ ]:
payload = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a cpp program with a blocked matrix multiplication algorithm"}
    ],
    "max_tokens": 1000,
    "temperature": 0.7,
    "stream": True,
}

response = requests.post(url, json=payload, headers=headers, timeout=60, stream=True)
response.raise_for_status()

print("🧠 Reasoning:", end=" ", flush=True)
reasoning = ""
final_content = ""

for line in response.iter_lines():
    if line:
        line = line.decode('utf-8')
        if line.startswith('data: '):
            data = line[6:]
            if data == '[DONE]':
                break
            
            chunk = json.loads(data)
            delta = chunk['choices'][0]['delta']
            
            if 'reasoning_content' in delta:
                text = delta['reasoning_content']
                reasoning += text
                print(text, end="", flush=True)